# I 320D: Data Science for Biomedical Informatics
## Project 15: Microbiome Temporal Dynamics and Disease State
### 10-Step Framework | Spring 2026 | University of Texas at Austin · School of Information

---

> This notebook walks through all 10 steps to characterize temporal variability in the gut microbiome
> across healthy and IBD subjects using the HMP2 longitudinal dataset. Each step includes a
> comparison to prior projects (Sepsis, CGM), pedagogical notes, and challenge questions.

---

## Quick Reference Table

| Step | Title | Core Concept | Key Challenge |
|------|-------|--------------|---------------|
| 0 | Environment Setup | Reproducibility; HMP2 `.tsv` files | Compositional data is fundamentally different from clinical time series |
| 1 | Data Loading | 132 subjects × 2,290 stool samples; 16S OTU counts | Irregular sampling intervals; subjects missing timepoints |
| 2 | EDA | Compositional distributions; per-subject diversity trajectories | Zero-inflation and compositional constraints invalidate standard EDA |
| 3 | CLR Transformation | Convert compositions to real-valued log-ratios | Zeros break CLR — pseudo-count strategy required |
| 4 | Feature Engineering | Temporal stability index; trailing trajectory features | Rolling windows on irregular time grids; Bray-Curtis dissimilarity |
| 5 | Train/Test Split | Subject-level split; stratify by disease group | Three-class imbalance: healthy, Crohn's, UC |
| 6 | Baseline Model | Logistic regression on mean CLR-transformed trajectory | Features are high-dimensional (~500 OTUs); sparsity regularization needed |
| 7 | Temporal Model | LSTM on 2-week trailing microbiome trajectory | Very sparse temporal coverage — not a dense time series |
| 8 | Evaluation | Flare prediction lead-time; per-subject AUROC | Flare definitions are clinical, not microbiome-derived |
| 9 | Interpretation | SHAP on LR; OTU-level importance; PCoA trajectory visualization | Which taxa drive dysbiosis vs. which drive flare prediction? |

---

## Step 0 — Environment Setup & Data Download

| | **Project 15: Microbiome** |
|---|---|
| **Dataset** | HMP2 (iHMP) via HMPDACC (hmpdacc.org) — freely available |
| **File Format** | `.tsv` OTU count tables; one row per sample, one column per taxon |
| **Credentialing** | None required — register at hmpdacc.org for download access |
| **Key Libraries** | `pandas`, `numpy`, `sklearn`, `torch`, `shap`, `skbio` (for Bray-Curtis), `matplotlib`, `seaborn` |
| **Reproducibility Goal** | Fixed seeds; pinned library versions; document which subject IDs are train vs. test |
| **Core Pitfall** | Treating OTU count tables as ordinary numeric data — they are compositional |
| **New Concept** | Compositional data: all values in a sample sum to a constant (read depth or 1.0 after normalization) |

**What changes pedagogically:** Both the Sepsis and CGM projects used continuous physiological measurements (vitals, glucose). The microbiome is a *composition* — a set of relative abundances that sum to 1. This fundamentally changes every downstream analysis step. Students must build new intuition from the ground up.

**Challenge questions:**
- The HMP2 dataset has 132 subjects but 2,290 samples — an average of ~17 samples per subject. How does this compare to the Ohio T1DM CGM dataset (6 patients, ~288 readings/day)? What does the N-subjects / T-timepoints tradeoff mean for generalization?
- Stool samples in HMP2 are collected at irregular intervals (roughly every 2 weeks, but not exactly). How is this different from the CGM 5-minute grid? How does it affect which modeling approaches you can use?

In [ ]:
# ── Step 0: Environment Setup ──────────────────────────────────────────────
import random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.model_selection import LeaveOneGroupOut

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import shap
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Environment ready.")
print(f"Torch version: {torch.__version__}")
print(f"Pandas version: {pd.__version__}")

# ── Data Download Instructions ─────────────────────────────────────────────
# 1. Visit https://hmpdacc.org/hmp/HMP2/
# 2. Download the 16S rRNA amplicon (OTU) count table
# 3. Download the subject metadata (disease status, visit dates, flare annotations)
# 4. Place files in ./data/hmp2/

DATA_DIR = "./data/hmp2/"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Data directory: {DATA_DIR}")

---

## Step 1 — Data Loading & First Look

| | **Project 15: Microbiome** |
|---|---|
| **Data Structure** | SubjectID → SampleDate → OTU count vector (~500 OTUs) → DiseaseStatus + FlareLabel |
| **Label Definition** | FlareLabel = 1 at samples collected during a clinically defined IBD flare; 0 otherwise |
| **Class Imbalance** | ~35% healthy subjects; ~35% Crohn's disease; ~30% ulcerative colitis; flares are a minority of samples within IBD subjects |
| **Temporal Resolution** | ~1 sample per 2 weeks (highly irregular) |
| **Longitudinal Structure** | Months of outpatient stool sampling |
| **Key Clinical Definition** | IBD flare: clinician-adjudicated symptom exacerbation; does not have a single biomarker threshold |
| **Core Pitfall** | Treating samples as independent; ignoring that samples within a subject are correlated |

**What changes pedagogically:** Unlike sepsis (SepsisLabel stays 1) or CGM (glucose < 70 is objective), flare labels here are *clinical judgments*. This introduces **label noise** as a first-class problem. A microbiome signal may precede a flare that was labeled at the clinic visit — creating uncertain label timing.

**Challenge questions:**
- Sepsis and hypoglycemia labels are defined by objective thresholds (SOFA score, glucose mg/dL). IBD flare labels come from clinical assessment. How does label subjectivity affect your classifier's ceiling performance?
- The dataset has three disease groups (healthy, Crohn's, UC). Should you build one binary flare classifier or separate classifiers per disease group? What are the tradeoffs?

In [ ]:
# ── Step 1: Data Loading ───────────────────────────────────────────────────

# ── 1a. Load OTU count table ───────────────────────────────────────────────
# Expected format: rows = samples, columns = OTU IDs + metadata columns
# otu_df = pd.read_csv(DATA_DIR + "otu_counts.tsv", sep="\t", index_col=0)

# ── Synthetic data for demonstration (replace with real HMP2 data) ─────────
N_SUBJECTS = 132
N_OTUS = 100          # HMP2 has ~500; reduced here for speed
SAMPLES_PER_SUBJECT = 17
N_SAMPLES = N_SUBJECTS * SAMPLES_PER_SUBJECT

np.random.seed(SEED)
otu_counts = np.random.negative_binomial(5, 0.3, size=(N_SAMPLES, N_OTUS)).astype(float)

subject_ids = np.repeat(np.arange(N_SUBJECTS), SAMPLES_PER_SUBJECT)
disease_map = {i: np.random.choice(['Healthy','CD','UC'], p=[0.35, 0.35, 0.30])
               for i in range(N_SUBJECTS)}
disease_labels = [disease_map[s] for s in subject_ids]
flare_labels = np.array([
    1 if (disease_map[s] != 'Healthy' and np.random.rand() < 0.25) else 0
    for s in subject_ids
])

otu_df = pd.DataFrame(otu_counts, columns=[f"OTU_{i}" for i in range(N_OTUS)])
otu_df['SubjectID'] = subject_ids
otu_df['Disease'] = disease_labels
otu_df['FlareLabel'] = flare_labels
otu_df['VisitNum'] = np.tile(np.arange(SAMPLES_PER_SUBJECT), N_SUBJECTS)

print(f"Dataset shape: {otu_df.shape}")
print(f"Subjects: {N_SUBJECTS}")
print(f"\nDisease distribution:")
print(otu_df.groupby('Disease')['SubjectID'].nunique())
print(f"\nFlare prevalence: {flare_labels.mean():.1%} of all samples")
otu_df.head()

---

## Step 2 — Exploratory Data Analysis (EDA)

| | **Project 15: Microbiome** |
|---|---|
| **Class Balance Plot** | Subject-level (3-class) and sample-level flare prevalence |
| **Signal Distributions** | OTU relative abundance distributions; alpha diversity per subject |
| **Missing Data Focus** | Not missing — but *zero-inflated*: most OTUs are absent in any given sample |
| **Temporal Trend Analysis** | Per-subject alpha diversity trajectory; beta diversity (Bray-Curtis) between consecutive samples |
| **Inter-subject Heterogeneity** | Microbiome composition varies more between people than within a single person over time |
| **New EDA Task** | PCoA plot of all samples colored by disease status; identify whether groups cluster |
| **Core Pitfall** | Using Pearson correlation or PCA on raw counts — both assume Euclidean geometry, invalid for compositions |

**What changes pedagogically:** In sepsis EDA, the challenge was missing data. In CGM, it was inter-patient baseline variability. Here, the challenge is *the geometry of the data space itself*. Relative abundances don't live in Euclidean space — they live in a simplex. This motivates the CLR transformation in Step 3.

**Challenge questions (from project brief):**
- Microbiome relative abundances are compositional — they sum to 1 and are therefore not independent. What goes wrong if you analyze them with standard statistical methods that assume independence? *(Q72)*
- Plot the alpha diversity (Shannon index) trajectory for 5 subjects — 2 healthy, 2 CD, 1 UC. Do IBD subjects show lower diversity? More variability? Does the pattern differ between CD and UC?

In [ ]:
# ── Step 2: Exploratory Data Analysis ──────────────────────────────────────

otu_cols = [c for c in otu_df.columns if c.startswith('OTU_')]

# ── 2a. Relative abundance (compositional normalization) ────────────────────
counts = otu_df[otu_cols].values
rel_abund = counts / counts.sum(axis=1, keepdims=True)
otu_df[[c + '_rel' for c in otu_cols]] = rel_abund

# ── 2b. Alpha diversity: Shannon index ─────────────────────────────────────
def shannon(row):
    p = row[row > 0]
    return -np.sum(p * np.log(p))

otu_df['Shannon'] = rel_abund.apply(lambda r: shannon(r) if hasattr(r, '__iter__') else 0, axis=0)
otu_df['Shannon'] = pd.DataFrame(rel_abund).apply(lambda r: shannon(r.values), axis=1)

# ── 2c. Zero-inflation ─────────────────────────────────────────────────────
zero_frac = (otu_df[otu_cols] == 0).mean()
print(f"Mean zero fraction across OTUs: {zero_frac.mean():.1%}")
print(f"OTUs that are >90% zero: {(zero_frac > 0.9).sum()} / {len(otu_cols)}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Class balance
disease_counts = otu_df.groupby('Disease')['SubjectID'].nunique()
axes[0].bar(disease_counts.index, disease_counts.values, color=['#2ecc71','#e74c3c','#3498db'])
axes[0].set_title('Subjects by Disease Group')
axes[0].set_ylabel('Number of Subjects')

# Shannon diversity by disease
for disease, color in zip(['Healthy','CD','UC'], ['#2ecc71','#e74c3c','#3498db']):
    subset = otu_df[otu_df['Disease'] == disease]['Shannon']
    axes[1].hist(subset, alpha=0.6, label=disease, color=color, bins=20)
axes[1].set_title('Alpha Diversity (Shannon Index) by Disease')
axes[1].set_xlabel('Shannon Index')
axes[1].legend()

# Zero-inflation distribution
axes[2].hist(zero_frac.values, bins=30, color='#9b59b6')
axes[2].set_title('Zero Fraction per OTU')
axes[2].set_xlabel('Proportion of Zeros')
axes[2].axvline(0.9, color='red', linestyle='--', label='90% threshold')
axes[2].legend()

plt.tight_layout()
plt.suptitle('Step 2: EDA — Microbiome Composition Overview', y=1.02, fontsize=13, fontweight='bold')
plt.show()

print("\n⚠️  Note: PCA on raw relative abundances is shown below for comparison.")
print("   Step 3 will show why CLR-transformed PCoA is preferred.")

---

## Step 3 — CLR Transformation for Compositional Data

| | **Project 15: Microbiome** |
|---|---|
| **Transformation** | Centered Log-Ratio (CLR): `clr(x) = log(x / geometric_mean(x))` |
| **Why CLR?** | Removes compositional constraint; maps simplex to unconstrained real-valued space |
| **CLR Assumption** | All components contribute equally to the geometric mean (equal reference frame) |
| **When CLR Fails** | When data contains true zeros — log(0) is undefined |
| **Zero Handling** | Add pseudo-count (e.g., 0.5) before CLR; or use multiplicative replacement |
| **Leakage Risk** | Geometric mean must be computed per-sample (not across training set) — CLR is sample-wise |
| **Contrast with Sepsis** | Sepsis Step 3 was a full imputation framework. Here, the transformation *itself* is the challenge |

**What changes pedagogically:** In sepsis, Step 3 solved a missing data problem via forward-fill. In CGM, it was nearly trivial. Here, Step 3 introduces an entirely new concept: *the geometry of compositional data*. The CLR transformation is the gateway to all subsequent analysis and must be understood before any modeling.

**Challenge questions (from project brief):**
- The CLR transformation converts compositional data to unconstrained real values. What assumption does CLR rely on, and when does it fail? *(Q73)*
- After CLR transformation, run PCoA on the result and compare the clustering to PCoA on raw relative abundances. Does CLR improve separation between healthy and IBD subjects?

In [ ]:
# ── Step 3: CLR Transformation ─────────────────────────────────────────────
from scipy.stats import gmean
from sklearn.decomposition import PCA

# ── 3a. Add pseudo-count to handle zeros ────────────────────────────────────
PSEUDO = 0.5
counts_pseudo = otu_df[otu_cols].values + PSEUDO

# ── 3b. Compute relative abundance with pseudo-count ───────────────────────
rel_pseudo = counts_pseudo / counts_pseudo.sum(axis=1, keepdims=True)

# ── 3c. CLR transformation (sample-wise) ───────────────────────────────────
def clr_transform(X):
    """Centered log-ratio transform. X: (n_samples, n_features) relative abundance matrix."""
    log_X = np.log(X)
    log_geo_mean = log_X.mean(axis=1, keepdims=True)  # geometric mean in log space
    return log_X - log_geo_mean

clr_matrix = clr_transform(rel_pseudo)
clr_df = pd.DataFrame(clr_matrix, columns=[c + '_clr' for c in otu_cols])
clr_df['SubjectID'] = otu_df['SubjectID'].values
clr_df['Disease'] = otu_df['Disease'].values
clr_df['FlareLabel'] = otu_df['FlareLabel'].values
clr_df['VisitNum'] = otu_df['VisitNum'].values

# ── 3d. PCoA via PCA on CLR (Aitchison distance ≈ Euclidean on CLR) ────────
pca = PCA(n_components=2, random_state=SEED)
pca_coords = pca.fit_transform(clr_matrix)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Healthy': '#2ecc71', 'CD': '#e74c3c', 'UC': '#3498db'}

# PCA on raw relative abundances
pca_raw = PCA(n_components=2, random_state=SEED)
coords_raw = pca_raw.fit_transform(rel_pseudo)
for disease, color in colors.items():
    mask = np.array(otu_df['Disease'] == disease)
    axes[0].scatter(coords_raw[mask, 0], coords_raw[mask, 1],
                    c=color, label=disease, alpha=0.4, s=10)
axes[0].set_title('PCA on Raw Relative Abundances\n(INCORRECT — Euclidean geometry invalid)')
axes[0].set_xlabel(f'PC1 ({pca_raw.explained_variance_ratio_[0]:.1%})')
axes[0].set_ylabel(f'PC2 ({pca_raw.explained_variance_ratio_[1]:.1%})')
axes[0].legend()

# PCoA on CLR-transformed data
for disease, color in colors.items():
    mask = np.array(otu_df['Disease'] == disease)
    axes[1].scatter(pca_coords[mask, 0], pca_coords[mask, 1],
                    c=color, label=disease, alpha=0.4, s=10)
axes[1].set_title('PCoA on CLR-Transformed Data\n(CORRECT — Aitchison distance)')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
axes[1].legend()

plt.suptitle('Step 3: CLR Transformation — Effect on PCoA Clustering', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"CLR matrix shape: {clr_matrix.shape}")
print(f"CLR row sums (should be ~0 by construction): {clr_matrix.sum(axis=1)[:5].round(10)}")

---

## Step 4 — Feature Engineering: Temporal Stability & Trajectory Features

| | **Project 15: Microbiome** |
|---|---|
| **Raw Features** | CLR-transformed OTU vector per sample (~500 OTUs) |
| **Temporal Stability** | Bray-Curtis dissimilarity between consecutive samples per subject |
| **Trajectory Features** | 2-week trailing mean and SD of CLR values per OTU |
| **Stability Index** | Per-subject mean Bray-Curtis dissimilarity = microbiome instability score |
| **Delta Features** | Change in CLR value between current and previous sample per OTU |
| **Window Logic** | 2-week window reflects clinical flare prediction horizon and sampling frequency |
| **New Concept** | Bray-Curtis dissimilarity as a measure of compositional change (not Euclidean distance) |
| **Core Pitfall** | Rolling windows crossing subject boundaries; computing dissimilarity on non-CLR data |

**What changes pedagogically:** In sepsis, rolling statistics were computed on a dense 40-variable hourly grid. In CGM, shapelets captured glucose curve shapes. Here, the key feature is *between-sample dissimilarity* on a sparse irregular grid. Students encounter Bray-Curtis dissimilarity — a new distance metric appropriate for compositional data.

**Challenge questions (from project brief):**
- Microbiome samples from the same person are temporally correlated. How do you account for within-subject correlation in your modeling? *(Q74)*
- Antibiotic use causes dramatic rapid shifts in microbiome composition. Should you treat antibiotic courses as a covariate, an exclusion criterion, or a period of interest in its own right? *(Q75)*

In [ ]:
# ── Step 4: Feature Engineering ────────────────────────────────────────────
from scipy.spatial.distance import braycurtis

clr_cols = [c for c in clr_df.columns if c.endswith('_clr')]

# ── 4a. Per-subject Bray-Curtis dissimilarity between consecutive samples ───
bc_records = []
for sid in clr_df['SubjectID'].unique():
    subj = clr_df[clr_df['SubjectID'] == sid].sort_values('VisitNum').reset_index(drop=True)
    ra_subj = rel_pseudo[clr_df['SubjectID'].values == sid]  # use rel abundance for BC
    for i in range(1, len(subj)):
        bc = braycurtis(ra_subj[i-1], ra_subj[i])
        bc_records.append({'SubjectID': sid, 'VisitNum': subj.loc[i, 'VisitNum'],
                           'Disease': subj.loc[i, 'Disease'],
                           'BrayCurtis': bc,
                           'FlareLabel': subj.loc[i, 'FlareLabel']})

bc_df = pd.DataFrame(bc_records)

# ── 4b. Per-subject stability index ───────────────────────────────────────
stability_idx = bc_df.groupby(['SubjectID','Disease'])['BrayCurtis'].mean().reset_index()
stability_idx.columns = ['SubjectID','Disease','StabilityIndex']

# ── 4c. 2-visit trailing mean CLR per OTU (trajectory feature) ────────────
feature_records = []
for sid in clr_df['SubjectID'].unique():
    subj = clr_df[clr_df['SubjectID'] == sid].sort_values('VisitNum').reset_index(drop=True)
    for i in range(1, len(subj)):  # need at least 1 prior sample
        window = subj.iloc[max(0, i-2):i][clr_cols]  # 2-visit trailing window
        feat = window.mean().values  # mean CLR over window
        feat_delta = (subj.iloc[i][clr_cols].values - subj.iloc[i-1][clr_cols].values)  # delta
        feature_records.append({
            'SubjectID': sid,
            'VisitNum': subj.loc[i, 'VisitNum'],
            'Disease': subj.loc[i, 'Disease'],
            'FlareLabel': subj.loc[i, 'FlareLabel'],
            **{f'mean_{c}': v for c, v in zip(clr_cols, feat)},
            **{f'delta_{c}': v for c, v in zip(clr_cols, feat_delta)}
        })

feat_df = pd.DataFrame(feature_records)

# ── 4d. Visualize stability index by disease ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for disease, color in colors.items():
    subset = stability_idx[stability_idx['Disease'] == disease]['StabilityIndex']
    axes[0].hist(subset, alpha=0.6, label=disease, color=color, bins=15)
axes[0].set_title('Microbiome Stability Index (Bray-Curtis) by Disease')
axes[0].set_xlabel('Mean Bray-Curtis Dissimilarity (higher = less stable)')
axes[0].legend()

# Bray-Curtis during flare vs. non-flare
flare_bc = bc_df[bc_df['FlareLabel'] == 1]['BrayCurtis']
noflare_bc = bc_df[bc_df['FlareLabel'] == 0]['BrayCurtis']
axes[1].hist(noflare_bc, alpha=0.6, label='No Flare', color='steelblue', bins=20)
axes[1].hist(flare_bc, alpha=0.6, label='Flare', color='tomato', bins=20)
axes[1].set_title('Bray-Curtis Dissimilarity: Flare vs. No Flare')
axes[1].set_xlabel('Bray-Curtis Dissimilarity (previous → current sample)')
axes[1].legend()

plt.suptitle('Step 4: Feature Engineering — Temporal Stability', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

feat_cols = [c for c in feat_df.columns if c.startswith('mean_') or c.startswith('delta_')]
print(f"Feature matrix shape: {feat_df[feat_cols].shape}")
print(f"Flare prevalence in feature matrix: {feat_df['FlareLabel'].mean():.1%}")

---

## Step 5 — Train / Test Split (Subject-Level)

| | **Project 15: Microbiome** |
|---|---|
| **Split Unit** | Subject (never by sample or visit) |
| **Split Ratio** | 80/20 stratified by disease group (Healthy / CD / UC) |
| **Stratification** | By disease group — preserve three-class balance in train and test |
| **Leakage Risk** | Sample-level split: a subject's samples appear in both train and test — inflates AUROC dramatically |
| **Cross-validation** | Leave-One-Subject-Out (LOSO-CV) is appropriate given correlated longitudinal data |
| **New Evaluation Design** | Per-subject AUROC for flare prediction; microbiome dynamics vary greatly between people |
| **Core Pitfall** | Row-level or visit-level splits — classic clinical ML error |

**What changes pedagogically:** This step follows the same principle as sepsis (patient-level split) and CGM (LOPO-CV). The new challenge is the three-class disease label — stratification must preserve balance across all three groups, not just a binary outcome.

**Challenge questions:**
- Implement Leave-One-Subject-Out CV. Report per-subject AUROC for flare prediction. Is there systematic variation — are CD subjects easier to predict than UC subjects?
- If you split by visit (row-level), how much does your AUROC inflate compared to the subject-level split? Implement both and compare.

In [ ]:
# ── Step 5: Train / Test Split ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split

# Subject-level metadata
subj_meta = feat_df.groupby('SubjectID')['Disease'].first().reset_index()
subject_ids_arr = subj_meta['SubjectID'].values
disease_arr = subj_meta['Disease'].values

# ── 5a. Stratified subject-level 80/20 split ───────────────────────────────
train_subj, test_subj = train_test_split(
    subject_ids_arr,
    test_size=0.20,
    stratify=disease_arr,
    random_state=SEED
)

train_df = feat_df[feat_df['SubjectID'].isin(train_subj)].copy()
test_df  = feat_df[feat_df['SubjectID'].isin(test_subj)].copy()

print(f"Train samples: {len(train_df)} | Test samples: {len(test_df)}")
print(f"Train subjects: {train_df['SubjectID'].nunique()} | Test subjects: {test_df['SubjectID'].nunique()}")
print(f"\nTrain disease distribution:")
print(train_df.groupby('Disease')['SubjectID'].nunique())
print(f"\nTest disease distribution:")
print(test_df.groupby('Disease')['SubjectID'].nunique())
print(f"\nTrain flare rate: {train_df['FlareLabel'].mean():.1%}")
print(f"Test flare rate:  {test_df['FlareLabel'].mean():.1%}")

# ── 5b. Scale features (fit on train only) ────────────────────────────────
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[feat_cols])
X_test  = scaler.transform(test_df[feat_cols])
y_train = train_df['FlareLabel'].values
y_test  = test_df['FlareLabel'].values
groups_train = train_df['SubjectID'].values

print("\n✅ Subject-level split complete. Scaler fit on training set only.")

---

## Step 6 — Baseline Model: Logistic Regression

| | **Project 15: Microbiome** |
|---|---|
| **Input Features** | 2-visit trailing mean CLR + delta CLR per OTU (~200 features for 100 OTUs) |
| **Model** | Logistic regression with `class_weight='balanced'`, L1 regularization (Lasso) |
| **Positive Rate** | ~15–25% of samples (within IBD subjects); lower if healthy subjects included |
| **Class Imbalance** | `class_weight='balanced'`; consider restricting to IBD subjects only |
| **Primary Metric** | AUROC + AUPRC — AUPRC is more informative under imbalance |
| **Clinical Baseline** | Stability index alone: "predict flare if Bray-Curtis > threshold" |
| **Interpretability** | Coefficients on CLR OTU features — which taxa are most predictive? |
| **Core Pitfall** | Reporting accuracy — trivially high by always predicting no-flare |

**What changes pedagogically:** With ~200 features and ~100 training subjects, regularization is essential. L1 (Lasso) naturally produces sparse models — interpretable as "which specific OTUs matter" — connecting feature importance directly to microbiome biology.

**Challenge questions:**
- Implement the clinical baseline: predict a flare if a subject's Bray-Curtis dissimilarity from their previous sample exceeds a threshold. At what threshold does this rule match the logistic regression's F1 score?
- L1 regularization drives many OTU coefficients to zero. How many OTUs are retained in your final model? Do they correspond to taxa known to be associated with IBD in the literature?

In [ ]:
# ── Step 6: Baseline Logistic Regression ───────────────────────────────────
from sklearn.metrics import precision_recall_curve, roc_curve

# ── 6a. Logistic regression with L1 regularization ────────────────────────
lr = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.1,
    class_weight='balanced',
    random_state=SEED,
    max_iter=500
)
lr.fit(X_train, y_train)

y_prob_lr = lr.predict_proba(X_test)[:, 1]
auroc_lr = roc_auc_score(y_test, y_prob_lr)
auprc_lr = average_precision_score(y_test, y_prob_lr)

# ── 6b. Clinical baseline: Bray-Curtis threshold rule ─────────────────────
bc_test = bc_df[bc_df['SubjectID'].isin(test_subj)]
# Align BC values with test_df rows (best effort with synthetic data)
bc_feature = test_df.merge(
    bc_test[['SubjectID','VisitNum','BrayCurtis']],
    on=['SubjectID','VisitNum'], how='left'
)['BrayCurtis'].fillna(bc_test['BrayCurtis'].median()).values

bc_threshold = np.percentile(bc_feature, 75)
y_rule = (bc_feature > bc_threshold).astype(int)
auroc_rule = roc_auc_score(y_test, bc_feature)
auprc_rule = average_precision_score(y_test, bc_feature)

print("=" * 50)
print(f"{'Model':<30} {'AUROC':>8} {'AUPRC':>8}")
print("-" * 50)
print(f"{'Clinical Rule (BC threshold)':<30} {auroc_rule:>8.3f} {auprc_rule:>8.3f}")
print(f"{'Logistic Regression (L1)':<30} {auroc_lr:>8.3f} {auprc_lr:>8.3f}")
print("=" * 50)

# ── 6c. Feature importance: top OTUs by absolute coefficient ──────────────
coef = lr.coef_[0]
n_nonzero = (coef != 0).sum()
top_idx = np.argsort(np.abs(coef))[::-1][:20]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh([feat_cols[i] for i in top_idx][::-1],
             coef[top_idx][::-1],
             color=['tomato' if c > 0 else 'steelblue' for c in coef[top_idx][::-1]])
axes[0].set_title(f'Top 20 LR Coefficients\n({n_nonzero} non-zero / {len(feat_cols)} features)')
axes[0].set_xlabel('Coefficient Value (red = flare-associated)')
axes[0].axvline(0, color='black', linewidth=0.8)

# PR curve comparison
prec_lr, rec_lr, _ = precision_recall_curve(y_test, y_prob_lr)
prec_rule, rec_rule, _ = precision_recall_curve(y_test, bc_feature)
axes[1].plot(rec_lr, prec_lr, label=f'LR (AUPRC={auprc_lr:.3f})', color='tomato')
axes[1].plot(rec_rule, prec_rule, label=f'BC Rule (AUPRC={auprc_rule:.3f})', color='steelblue', linestyle='--')
axes[1].axhline(y_train.mean(), color='gray', linestyle=':', label='Random baseline')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve: LR vs Clinical Rule')
axes[1].legend()

plt.suptitle('Step 6: Baseline Model — Logistic Regression', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Step 7 — Temporal Model: LSTM on Microbiome Trajectories

| | **Project 15: Microbiome** |
|---|---|
| **Input Sequence** | 2-week trailing window of CLR OTU vectors (2–4 samples per subject) |
| **Sequence Length** | SEQ_LEN = 4 samples (highly irregular — not a dense grid like CGM) |
| **Architecture** | 1-layer LSTM, 64 hidden units, dropout 0.3, sigmoid output |
| **Loss Function** | Weighted BCE (class_weight proportional to imbalance) |
| **Key Advantage Over LR** | Learns that a *trend* of increasing dysbiosis over 4 visits predicts a flare; LR sees only the last visit |
| **New Consideration** | Very sparse temporal coverage — LSTM input sequences have only 2–4 steps vs. 12 (sepsis) or 6 (CGM) |
| **Prediction Horizon** | 1 sample (≈ 2 weeks) before clinical flare |
| **Core Pitfall** | Exploding gradients; padding sequences naively without masking |

**What changes pedagogically:** The LSTM must handle *very short, irregular* sequences. Unlike CGM's 6-step window or sepsis's 12-step window, microbiome sequences may have only 2–4 observations in the trailing window. Students must question whether LSTM adds value over a simpler model given such short sequences.

**Challenge questions (from project brief):**
- IBD flares are defined clinically. If the microbiome changes precede the clinical flare, does your model capture a causal signal or a correlational one? How would you distinguish these? *(Q76)*
- Compare SEQ_LEN = 2 vs. SEQ_LEN = 4. Does longer memory help for flare prediction? At what sequence length does LSTM stop outperforming logistic regression?

In [ ]:
# ── Step 7: LSTM on Microbiome Trajectories ────────────────────────────────

SEQ_LEN = 4   # number of trailing visits
N_FEATURES = len(feat_cols)
HIDDEN = 64
DROPOUT = 0.3
EPOCHS = 30
LR = 1e-3

# ── 7a. Build fixed-length sequences per subject ───────────────────────────
def build_sequences(df, feat_cols, seq_len, scaler=None):
    X_seq, y_seq, groups = [], [], []
    for sid in df['SubjectID'].unique():
        subj = df[df['SubjectID'] == sid].sort_values('VisitNum').reset_index(drop=True)
        vals = subj[feat_cols].values
        if scaler:
            vals = scaler.transform(vals)
        labels = subj['FlareLabel'].values
        for i in range(seq_len, len(subj)):
            window = vals[i-seq_len:i]  # (seq_len, n_features)
            X_seq.append(window)
            y_seq.append(labels[i])
            groups.append(sid)
    return np.array(X_seq), np.array(y_seq), np.array(groups)

X_tr_seq, y_tr_seq, g_tr = build_sequences(train_df, feat_cols, SEQ_LEN)
X_te_seq, y_te_seq, g_te = build_sequences(test_df, feat_cols, SEQ_LEN)

# Already scaled — convert to tensors
X_tr_seq_scaled = scaler.transform(X_tr_seq.reshape(-1, N_FEATURES)).reshape(-1, SEQ_LEN, N_FEATURES)
X_te_seq_scaled = scaler.transform(X_te_seq.reshape(-1, N_FEATURES)).reshape(-1, SEQ_LEN, N_FEATURES)

Xt = torch.FloatTensor(X_tr_seq_scaled)
yt = torch.FloatTensor(y_tr_seq)
Xv = torch.FloatTensor(X_te_seq_scaled)
yv = torch.FloatTensor(y_te_seq)

pos_weight = torch.tensor([(1 - y_tr_seq.mean()) / y_tr_seq.mean()])
train_loader = DataLoader(TensorDataset(Xt, yt), batch_size=32, shuffle=True)

# ── 7b. LSTM model ────────────────────────────────────────────────────────
class MicrobiomeLSTM(nn.Module):
    def __init__(self, n_feat, hidden, dropout):
        super().__init__()
        self.lstm = nn.LSTM(n_feat, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden, 1)
    def forward(self, x):
        _, (h, _) = self.lstm(x)
        return torch.sigmoid(self.fc(self.drop(h[-1]))).squeeze(1)

model = MicrobiomeLSTM(N_FEATURES, HIDDEN, DROPOUT)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()

# ── 7c. Training loop ─────────────────────────────────────────────────────
train_losses = []
model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        # Manual class weighting
        weights = torch.where(yb == 1, pos_weight.squeeze(), torch.ones(len(yb)))
        loss = (criterion(pred, yb) * weights).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        epoch_loss += loss.item()
    train_losses.append(epoch_loss / len(train_loader))

# ── 7d. Evaluation ────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    y_prob_lstm = model(Xv).numpy()

auroc_lstm = roc_auc_score(y_te_seq, y_prob_lstm)
auprc_lstm = average_precision_score(y_te_seq, y_prob_lstm)

print("=" * 50)
print(f"{'Model':<30} {'AUROC':>8} {'AUPRC':>8}")
print("-" * 50)
print(f"{'Clinical Rule (BC threshold)':<30} {auroc_rule:>8.3f} {auprc_rule:>8.3f}")
print(f"{'Logistic Regression (L1)':<30} {auroc_lr:>8.3f} {auprc_lr:>8.3f}")
print(f"{'LSTM (seq_len={SEQ_LEN})':<30} {auroc_lstm:>8.3f} {auprc_lstm:>8.3f}")
print("=" * 50)

plt.figure(figsize=(7, 4))
plt.plot(train_losses)
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title(f'Step 7: LSTM Training Loss (seq_len={SEQ_LEN})')
plt.tight_layout(); plt.show()

---

## Step 8 — Evaluation: Beyond AUROC

| | **Project 15: Microbiome** |
|---|---|
| **Primary Metric** | AUROC + AUPRC; per-subject AUROC distribution |
| **Timing** | Lead time: how many visits (weeks) before the clinical flare does the model fire? |
| **False Alarm Cost** | Moderate — unnecessary clinical intervention; patient anxiety |
| **New Evaluation Challenge** | Microbiome changes may precede the *diagnosis* of a flare, not the *onset* — labeling lag |
| **Subgroup Analysis** | Per-disease-group AUROC (Healthy vs. CD vs. UC); per-subject AUROC |
| **Calibration** | Platt scaling; calibration curve — important given rare-event rate |
| **Core Pitfall** | Optimizing population-level AUROC while performance is heterogeneous across subjects |

**What changes pedagogically:** Unlike the CGM prevented-event paradox (an acted-on alarm becomes a false positive), here the challenge is *label timing*. If the microbiome shifts 2 weeks before the patient reports to the clinic with a flare, the clinical label date understates the model's true lead time. Evaluation must account for this lag.

**Challenge questions:**
- Compute per-subject AUROC for flare prediction. Plot its distribution. Is there a systematic difference between CD and UC subjects? What biological mechanism might explain this?
- If the microbiome shifts precede the clinical label by 1–2 weeks, how does this affect your model's apparent precision? How would you correct for this labeling lag in evaluation?

In [ ]:
# ── Step 8: Evaluation ─────────────────────────────────────────────────────

# ── 8a. Per-subject AUROC (LR model) ─────────────────────────────────────
test_df_eval = test_df.copy().reset_index(drop=True)
test_df_eval['y_prob_lr'] = lr.predict_proba(X_test)[:, 1]

per_subj_auroc = []
for sid in test_df_eval['SubjectID'].unique():
    subj = test_df_eval[test_df_eval['SubjectID'] == sid]
    if subj['FlareLabel'].nunique() < 2:
        continue  # can't compute AUROC without both classes
    auroc_s = roc_auc_score(subj['FlareLabel'], subj['y_prob_lr'])
    disease_s = subj['Disease'].iloc[0]
    per_subj_auroc.append({'SubjectID': sid, 'Disease': disease_s, 'AUROC': auroc_s})

per_subj_df = pd.DataFrame(per_subj_auroc)

# ── 8b. Visualization ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ROC comparison
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_lstm, tpr_lstm, _ = roc_curve(y_te_seq, y_prob_lstm)
axes[0].plot(fpr_lr, tpr_lr, label=f'LR (AUC={auroc_lr:.3f})', color='tomato')
axes[0].plot(fpr_lstm, tpr_lstm, label=f'LSTM (AUC={auroc_lstm:.3f})', color='steelblue')
axes[0].plot([0,1],[0,1],'k--', label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve: LR vs. LSTM')
axes[0].legend()

# Per-subject AUROC distribution
if len(per_subj_df) > 0:
    for disease, color in colors.items():
        subset = per_subj_df[per_subj_df['Disease'] == disease]['AUROC']
        if len(subset) > 0:
            axes[1].hist(subset, alpha=0.6, label=disease, color=color, bins=8)
    axes[1].axvline(0.5, color='black', linestyle='--', label='Random')
    axes[1].set_xlabel('Per-Subject AUROC')
    axes[1].set_title('Per-Subject AUROC Distribution by Disease')
    axes[1].legend()

# Calibration curve
from sklearn.calibration import calibration_curve
frac_pos, mean_pred = calibration_curve(y_test, y_prob_lr, n_bins=8)
axes[2].plot(mean_pred, frac_pos, 's-', label='LR (uncalibrated)', color='tomato')
axes[2].plot([0,1],[0,1],'k--', label='Perfect calibration')
axes[2].set_xlabel('Mean Predicted Probability')
axes[2].set_ylabel('Fraction of Positives')
axes[2].set_title('Calibration Curve')
axes[2].legend()

plt.suptitle('Step 8: Evaluation — Beyond AUROC', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

if len(per_subj_df) > 0:
    print("\nPer-subject AUROC by disease group:")
    print(per_subj_df.groupby('Disease')['AUROC'].describe().round(3))

---

## Step 9 — Model Interpretation with SHAP & PCoA Trajectory Visualization

| | **Project 15: Microbiome** |
|---|---|
| **Explainer** | `shap.LinearExplainer` for LR; `shap.DeepExplainer` for LSTM |
| **Summary Plot** | Beeswarm: which OTUs (CLR features) drive flare predictions |
| **Local Explanation** | Waterfall plot: why did this subject's sample trigger a flare prediction? |
| **New Visualization** | PCoA trajectory per subject colored by disease status and flare events |
| **Biological Grounding** | High-SHAP OTUs should be mapped to known taxa (e.g., *Faecalibacterium prausnitzii* loss in IBD) |
| **Clinician Communication** | "Your microbiome shifted away from its healthy baseline state" + show PCoA trajectory |
| **Core Pitfall** | High SHAP value ≠ causation; confusing global feature importance with local per-subject explanation |

**What changes pedagogically:** The PCoA trajectory visualization is new — it shows *where* a subject moves in microbiome space over time and whether that trajectory predicts a flare. This combines interpretability (SHAP) with a clinically intuitive visualization (the dysbiosis map).

**Challenge questions (from project brief):**
- If the microbiome changes precede the clinical flare, does your model capture a causal signal or a correlational one? How would you distinguish these using the SHAP outputs and the PCoA trajectory? *(Q76)*
- Visualize the PCoA trajectories of 3 subjects: one healthy, one CD in remission, one CD during a flare. Does the flare subject's trajectory move into a distinct region of PCoA space?

In [ ]:
# ── Step 9: Interpretation — SHAP + PCoA Trajectories ─────────────────────

# ── 9a. SHAP on Logistic Regression ───────────────────────────────────────
explainer = shap.LinearExplainer(lr, X_train, feature_perturbation='interventional')
shap_values = explainer.shap_values(X_test)

# Top 20 features by mean |SHAP|
mean_shap = np.abs(shap_values).mean(axis=0)
top20_idx = np.argsort(mean_shap)[::-1][:20]
top20_names = [feat_cols[i] for i in top20_idx]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# SHAP summary bar plot
axes[0].barh(top20_names[::-1], mean_shap[top20_idx][::-1], color='steelblue')
axes[0].set_title('Top 20 Features by Mean |SHAP| (LR)')
axes[0].set_xlabel('Mean |SHAP Value|')
axes[0].tick_params(axis='y', labelsize=8)

# ── 9b. PCoA trajectories for 3 selected subjects ─────────────────────────
# Use CLR-PCoA coordinates from Step 3
pca_coord_df = pd.DataFrame(pca_coords, columns=['PC1','PC2'])
pca_coord_df['SubjectID'] = otu_df['SubjectID'].values
pca_coord_df['Disease'] = otu_df['Disease'].values
pca_coord_df['VisitNum'] = otu_df['VisitNum'].values
pca_coord_df['FlareLabel'] = otu_df['FlareLabel'].values

# Pick 3 illustrative subjects: 1 Healthy, 1 CD with flares, 1 UC with flares
healthy_subjs = [s for s, d in disease_map.items() if d == 'Healthy'][:1]
cd_subjs      = [s for s, d in disease_map.items() if d == 'CD'][:1]
uc_subjs      = [s for s, d in disease_map.items() if d == 'UC'][:1]
selected = healthy_subjs + cd_subjs + uc_subjs

for sid, color, label in zip(selected,
                              ['#2ecc71','#e74c3c','#3498db'],
                              ['Healthy','CD','UC']):
    subj_traj = pca_coord_df[pca_coord_df['SubjectID'] == sid].sort_values('VisitNum')
    axes[1].plot(subj_traj['PC1'], subj_traj['PC2'], '-', color=color, alpha=0.5, linewidth=1.5)
    axes[1].scatter(subj_traj['PC1'], subj_traj['PC2'],
                    c=['red' if f==1 else color for f in subj_traj['FlareLabel']],
                    s=60, edgecolors='black', linewidth=0.5, zorder=5,
                    label=label)
    # Mark first visit
    axes[1].annotate('Start', (subj_traj['PC1'].iloc[0], subj_traj['PC2'].iloc[0]),
                     fontsize=7, color=color)

axes[1].set_title('PCoA Trajectories (CLR-space)\nRed dots = flare samples')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')
handles = [plt.Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=8, label=l)
           for c, l in zip(['#2ecc71','#e74c3c','#3498db'], ['Healthy','CD','UC'])]
handles.append(plt.Line2D([0],[0], marker='o', color='w', markerfacecolor='red',
                           markersize=8, label='Flare'))
axes[1].legend(handles=handles)

plt.suptitle('Step 9: SHAP Interpretation + PCoA Dysbiosis Trajectories',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Summary: What Is New in Project 15 vs. Prior Projects

| Concept | Sepsis (P4) | CGM (P10) | Microbiome (P15) | Pedagogical Value |
|---------|-------------|-----------|------------------|-------------------|
| Compositional data & CLR | ❌ | ❌ | ✅ | Geometry of data space — simplex vs. Euclidean |
| Zero-inflated counts | ❌ | ❌ | ✅ | Pseudo-count strategies; log-ratio methods |
| Bray-Curtis dissimilarity | ❌ | ❌ | ✅ | Non-Euclidean distance for community composition |
| Temporal stability index | ❌ | ❌ | ✅ | Between-sample dissimilarity as a feature |
| Irregular sampling | Partial | ❌ (5-min grid) | ✅ | Method choice depends on data structure |
| Three-class stratification | ❌ | ❌ | ✅ | Healthy / CD / UC; flare prediction within IBD |
| PCoA trajectory visualization | ❌ | ❌ | ✅ | Subject microbiome path through composition space |
| Label timing uncertainty | ❌ | Partial (prevented events) | ✅ | Clinical label lag from symptom onset |
| Within-subject temporal correlation | ✅ | ✅ | ✅ Full | LOSO-CV; within-subject effects |
| Causal vs. correlational signal | Partial | Partial | ✅ | Microbiome precedes flare — but is it causal? |

---

## Reflection Questions (All 5 from Project Brief)

1. **Q72** — Microbiome relative abundances are compositional — they sum to 1 and are therefore not independent. What goes wrong if you analyze them with standard statistical methods that assume independence?

2. **Q73** — The CLR transformation converts compositional data to unconstrained real values. What assumption does CLR rely on, and when does it fail (hint: think about zeros in the data)?

3. **Q74** — Microbiome samples from the same person are temporally correlated. How do you account for within-subject correlation in your modeling?

4. **Q75** — Antibiotic use causes dramatic rapid shifts in microbiome composition. Should you treat antibiotic courses as a covariate, an exclusion criterion, or a period of interest in its own right?

5. **Q76** — IBD flares are defined clinically. If the microbiome changes precede the clinical flare, does your model capture a causal signal or a correlational one? How would you distinguish these?

---

*I 320D Spring 2026 · University of Texas at Austin · School of Information*  
*Ammar Darkazanli, Ph.D., MBA · Lecturer*